In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
df=pd.read_excel('../Bariatric Project Study Data 2025_mod.xlsx', sheet_name='Sheet1')
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, precision_score, recall_score, f1_score
)
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
core_X_data=df[['gender', 'age','height', 'weight', 'bmi', 'family_hist_cnt', 'chronic_meds_cnt','procedure_category','antibiotics']].copy()
cm_cols= [col for col in df.columns if col.startswith('CM_')]

cm_data = df[cm_cols].fillna(0).astype(int)
cm_data
zero_only = [col for col in cm_data.columns if (cm_data[col] == 0).all()]
cm_data = cm_data.drop(columns=zero_only)

# finally, join the filtered CM_ data back into your core set
core_X_data = core_X_data.join(cm_data)
core_X_data.fillna(0, inplace=True)
print(core_X_data.isna().sum())  # should all be zero   

# inspect result
print("Dropped CM columns:", zero_only)
print("Remaining columns in CM data:", cm_data.columns.tolist())
# 1) Grab all the antibiotic dummy columns
ab_cols = [c for c in core_X_data.columns if c.startswith('antibiotics')]

# 2) Count zeros in each column
for col in ab_cols:
    n_zeros = (core_X_data[col] == 0).sum()
    print(f"{col}: {n_zeros} zeros out of {len(core_X_data)} rows")

# 3) (Bonus) How many rows have *no* antibiotic flagged?
no_ab_rows = (core_X_data[ab_cols].sum(axis=1) == 0).sum()
print(f"\nRows where ALL antibiotic dummies are 0 (i.e. original 0): {no_ab_rows}")

gender                0
age                   0
height                0
weight                0
bmi                   0
family_hist_cnt       0
chronic_meds_cnt      0
procedure_category    0
antibiotics           0
CM_AIDS               0
CM_ANEMDEF            0
CM_ARTH               0
CM_CHF                0
CM_DEPRESS            0
CM_DM                 0
CM_DMCX               0
CM_HTN_C              0
CM_HYPOTHY            0
CM_LIVER              0
CM_OBESE              0
CM_PSYCH              0
CM_SMOKE              0
CM_APNEA              0
CM_CHOLSTRL           0
CM_OSTARTH            0
CM_HPLD               0
dtype: int64
Dropped CM columns: ['CM_ALCOHOL', 'CM_BLDLOSS', 'CM_CHRNLUNG', 'CM_COAG', 'CM_DRUG', 'CM_LYMPH', 'CM_LYTES', 'CM_METS', 'CM_NEURO', 'CM_PARA', 'CM_PERIVASC', 'CM_PULMCIRC', 'CM_RENLFAIL', 'CM_TUMOR', 'CM_ULCER', 'CM_VALVE', 'CM_WGHTLOSS']
Remaining columns in CM data: ['CM_AIDS', 'CM_ANEMDEF', 'CM_ARTH', 'CM_CHF', 'CM_DEPRESS', 'CM_DM', 'CM_DMCX', 'CM_HTN_C', 

In [ ]:
core_X_data.shape

(344, 26)

In [ ]:
cm_to_remove = [
    "CM_AIDS",
    "CM_ANEMDEF",
    "CM_ARTH",
    "CM_CHF",
    "CM_DEPRESS",
    "CM_HYPOTHY",
    "CM_PSYCH",
    "CM_SMOKE"
]
# Remove specified columns
core_X_data = core_X_data.drop(columns=cm_to_remove)

In [ ]:
# Identify all complication-related columns
comp_cols = [col for col in df.columns if 'comp' in col.lower()]

In [ ]:
comp_cols

['post_op_complication',
 'days_30_complication',
 'days_30_plus_complication',
 'minor_complication',
 'major_complication',
 'minor_comp_month_1',
 'complications',
 'surgery_complication',
 'post_op_complication.1',
 'days_30_complication.1',
 'days_30_plus_complication.1']

In [ ]:
complication_data = df[['minor_complication', 'major_complication']].copy()
complication_data.head(1000)

,minor_complication,major_complication
0,0,0
1,0,0
2,1,0
3,0,0
4,0,0
...,...,...
339,0,0
340,0,0
341,1,0
342,0,0


In [ ]:
X = core_X_data.copy()
y_major = complication_data['major_complication']
y_minor = complication_data['minor_complication']

In [ ]:
from sklearn.model_selection import train_test_split
# Handle categorical variables first
categorical_cols = X.select_dtypes(include=['object']).columns
X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)


# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y_minor, test_size=0.2, random_state=42, stratify=y_minor
)

print("Class distribution in training set:")
print(f"No complications: {(y_train==0).sum()} ({(y_train==0).sum()/len(y_train)*100:.1f}%)")
print(f"Complications: {(y_train==1).sum()} ({(y_train==1).sum()/len(y_train)*100:.1f}%)")
print(f"\nClass distribution in test set:")
print(f"No complications: {(y_test==0).sum()} ({(y_test==0).sum()/len(y_test)*100:.1f}%)")
print(f"Complications: {(y_test==1).sum()} ({(y_test==1).sum()/len(y_test)*100:.1f}%)")
X_encoded.head()

Class distribution in training set:
No complications: 253 (92.0%)
Complications: 22 (8.0%)

Class distribution in test set:
No complications: 64 (92.8%)
Complications: 5 (7.2%)


,age,height,weight,bmi,family_hist_cnt,chronic_meds_cnt,CM_DM,CM_DMCX,CM_HTN_C,CM_LIVER,...,procedure_category_BPD -DS,procedure_category_Mini gastric bypass (OAGB),procedure_category_RYGBP,procedure_category_SADI,procedure_category_Sleeve,antibiotics_Augmentin,antibiotics_Clindamycin,antibiotics_Invanz,antibiotics_Kefsol,antibiotics_Rocephin
0,50,154,146.0,61.56,0,0,1,1,1,0,...,False,False,False,True,False,False,False,False,True,False
1,52,168,96.0,34.00,0,0,1,0,1,0,...,False,False,False,False,True,False,False,False,True,False
2,23,163,143.0,53.82,0,0,0,0,0,0,...,False,False,False,True,False,False,False,False,True,False
3,23,176,120.0,38.74,0,0,0,0,0,0,...,False,False,False,False,True,False,False,False,True,False
4,57,162,112.0,42.68,0,7,0,0,0,0,...,False,False,False,True,False,False,False,False,True,False


In [27]:

y_any = ((y_minor == 1) | (y_major == 1)).astype(int)
mask  = (y_any == 1)


if isinstance(X, pd.DataFrame):
    X_comp = X.iloc[mask.values]
else:
    X_comp = X[mask]

In [30]:
y_type = np.asarray(y_major)[mask]

In [ ]:
X= X_encoded.copy()

In [31]:
from sklearn.ensemble        import BaggingClassifier
from sklearn.linear_model   import LogisticRegression
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score

# 1) Base model: regularized logistic
base_clf = LogisticRegression(
    penalty="l2",
    C=0.1,
    class_weight="balanced",
    solver="liblinear",
    random_state=42
)

# 2) Bagging wrapper (use `estimator=` not `base_estimator=`)
bag_clf = BaggingClassifier(
    estimator=base_clf,
    n_estimators=100,
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)

# 3) Evaluate with repeated stratified CV
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)
scores = cross_val_score(bag_clf, X_comp, y_type, cv=cv, scoring="recall", n_jobs=-1)

print(f"Recall: {scores.mean():.2f} ± {scores.std():.2f}")


Recall: 0.64 ± 0.40


In [ ]:
from joblib import load
model_any = load("backend/SMOTE_logReg_risk_model.pkl")

# Stage-2: major vs minor, bagged logistic
from sklearn.ensemble import BaggingClassifier
from sklearn.linear_model import LogisticRegression

base_clf = LogisticRegression(
    penalty="l2", C=0.1, class_weight="balanced",
    solver="liblinear", random_state=42
)
model_type = BaggingClassifier(
    estimator=base_clf, n_estimators=100,
    bootstrap=True, random_state=42, n_jobs=-1
)
model_type.fit(X_comp, y_type)  # X_comp, y_type are your 34 complication cases


BaggingClassifier(estimator=LogisticRegression(C=0.1, class_weight='balanced',
                                               random_state=42,
                                               solver='liblinear'),
                  n_estimators=100, n_jobs=-1, random_state=42)

In [34]:
import random
import numpy as np

# — assume you’ve already loaded df, X, y_minor, y_major —
# — and trained:
#     model_any   : predicts P(any complication)
#     model_type  : predicts P(major | complication) on X_comp/y_type

# 1) Pick a random patient index
idx = random.choice(df.index.tolist())

# 2) Extract their features for prediction
#    keep as a 2D array / DataFrame slice
x_new = X.loc[[idx]] if hasattr(X, "loc") else X[np.newaxis, idx]
# Remove height and weight from bias model
#    (if you want to predict risk for a patient with no height/weight)

x_new_bias=X.loc[[idx]] if hasattr(X, "loc") else X[np.newaxis, idx]
x_new_bias = x_new_bias.drop(columns=['height', 'weight'])

# 3) Stage 1: P(any complication)
p_any = model_any.predict_proba(x_new_bias)[0, 1]

# 4) Stage 2: P(major | complication)
#    if you only trained model_type on complication cases,
#    you might want to guard—but here we just call it
p_given = model_type.predict_proba(x_new)[0, 1]

# 5) Joint probabilities
p_major = p_any * p_given
p_minor = p_any * (1 - p_given)

# 6) Reveal predictions and the true outcome
true_minor = y_minor[idx]
true_major = y_major[idx]

print(f"Random patient index: {idx}")
print("Features:", x_new.to_dict(orient="records")[0])
print(f"P(any complication) = {p_any:.3f}")
print(f"P(major | complication) = {p_given:.3f}")
print(f"→ P(major) = {p_major:.3f}, P(minor) = {p_minor:.3f}")
print(f"True outcome: minor_complication = {true_minor}, major_complication = {true_major}")


Random patient index: 176
Features: {'age': 42, 'height': 165, 'weight': 144.0, 'bmi': 52.89, 'family_hist_cnt': 0, 'chronic_meds_cnt': 7, 'CM_DM': 1, 'CM_DMCX': 1, 'CM_HTN_C': 1, 'CM_LIVER': 0, 'CM_OBESE': 1, 'CM_APNEA': 0, 'CM_CHOLSTRL': 1, 'CM_OSTARTH': 1, 'CM_HPLD': 1, 'gender_Male': False, 'procedure_category_BPD -DS': False, 'procedure_category_Mini gastric bypass (OAGB)': False, 'procedure_category_RYGBP': False, 'procedure_category_SADI': False, 'procedure_category_Sleeve': False, 'antibiotics_Augmentin': False, 'antibiotics_Clindamycin': False, 'antibiotics_Invanz': False, 'antibiotics_Kefsol': True, 'antibiotics_Rocephin': False}
P(any complication) = 0.430
P(major | complication) = 0.088
→ P(major) = 0.038, P(minor) = 0.392
True outcome: minor_complication = 0, major_complication = 0


In [35]:
import pickle

# Replace 'model_given' with your actual model or object
with open('model_type.pkl', 'wb') as file:
    pickle.dump(model_type, file)


In [36]:
X.columns

Index(['age', 'height', 'weight', 'bmi', 'family_hist_cnt', 'chronic_meds_cnt',
       'CM_DM', 'CM_DMCX', 'CM_HTN_C', 'CM_LIVER', 'CM_OBESE', 'CM_APNEA',
       'CM_CHOLSTRL', 'CM_OSTARTH', 'CM_HPLD', 'gender_Male',
       'procedure_category_BPD -DS',
       'procedure_category_Mini gastric bypass (OAGB)',
       'procedure_category_RYGBP', 'procedure_category_SADI',
       'procedure_category_Sleeve', 'antibiotics_Augmentin',
       'antibiotics_Clindamycin', 'antibiotics_Invanz', 'antibiotics_Kefsol',
       'antibiotics_Rocephin'],
      dtype='object')

In [37]:
import joblib

minor_major_model = load("model_type.pkl")

p_any = minor_major_model.predict_proba(x_new)[0, 1]
p_any

0.08797550009617215

In [38]:
{'age': np.float64(57.0), 'height': 0.0, 'weight': 0.0, 'bmi': np.float64(49.45), 'family_hist_cnt': np.float64(0.0), 'chronic_meds_cnt': np.float64(0.0), 'CM_DM': np.float64(1.0), 'CM_DMCX': np.float64(1.0), 'CM_HTN_C': np.float64(1.0), 'CM_LIVER': np.float64(0.0), 'CM_OBESE': np.float64(1.0), 'CM_APNEA': np.float64(2.0), 'CM_CHOLSTRL': np.float64(1.0), 'CM_OSTARTH': np.float64(0.0), 'CM_HPLD': np.float64(1.0), 'gender_Male': np.float64(1.0), 'procedure_category_BPD -DS': np.int64(0), 'procedure_category_Mini gastric bypass (OAGB)': np.int64(0), 'procedure_category_RYGBP': np.int64(0), 'procedure_category_SADI': np.int64(0), 'procedure_category_Sleeve': np.int64(0), 'antibiotics_Augmentin': np.int64(1), 'antibiotics_Clindamycin': np.int64(0), 'antibiotics_Invanz': np.int64(0), 'antibiotics_Kefsol': np.int64(0), 'antibiotics_Rocephin': np.int64(0)}

{'age': 57.0,
 'height': 0.0,
 'weight': 0.0,
 'bmi': 49.45,
 'family_hist_cnt': 0.0,
 'chronic_meds_cnt': 0.0,
 'CM_DM': 1.0,
 'CM_DMCX': 1.0,
 'CM_HTN_C': 1.0,
 'CM_LIVER': 0.0,
 'CM_OBESE': 1.0,
 'CM_APNEA': 2.0,
 'CM_CHOLSTRL': 1.0,
 'CM_OSTARTH': 0.0,
 'CM_HPLD': 1.0,
 'gender_Male': 1.0,
 'procedure_category_BPD -DS': 0,
 'procedure_category_Mini gastric bypass (OAGB)': 0,
 'procedure_category_RYGBP': 0,
 'procedure_category_SADI': 0,
 'procedure_category_Sleeve': 0,
 'antibiotics_Augmentin': 1,
 'antibiotics_Clindamycin': 0,
 'antibiotics_Invanz': 0,
 'antibiotics_Kefsol': 0,
 'antibiotics_Rocephin': 0}

In [39]:
{'age': 57, 'height': 164, 'weight': 133.0, 'bmi': 49.45, 'family_hist_cnt': 0, 'chronic_meds_cnt': 0, 'CM_DM': 1, 'CM_DMCX': 1, 'CM_HTN_C': 1, 'CM_LIVER': 0, 'CM_OBESE': 1, 'CM_APNEA': 2, 'CM_CHOLSTRL': 1, 'CM_OSTARTH': 1, 'CM_HPLD': 1, 'gender_Male': True, 'procedure_category_BPD -DS': False, 'procedure_category_Mini gastric bypass (OAGB)': False, 'procedure_category_RYGBP': True, 'procedure_category_SADI': False, 'procedure_category_Sleeve': False, 'antibiotics_Augmentin': True, 'antibiotics_Clindamycin': False, 'antibiotics_Invanz': False, 'antibiotics_Kefsol': False, 'antibiotics_Rocephin': False}

{'age': 57,
 'height': 164,
 'weight': 133.0,
 'bmi': 49.45,
 'family_hist_cnt': 0,
 'chronic_meds_cnt': 0,
 'CM_DM': 1,
 'CM_DMCX': 1,
 'CM_HTN_C': 1,
 'CM_LIVER': 0,
 'CM_OBESE': 1,
 'CM_APNEA': 2,
 'CM_CHOLSTRL': 1,
 'CM_OSTARTH': 1,
 'CM_HPLD': 1,
 'gender_Male': True,
 'procedure_category_BPD -DS': False,
 'procedure_category_Mini gastric bypass (OAGB)': False,
 'procedure_category_RYGBP': True,
 'procedure_category_SADI': False,
 'procedure_category_Sleeve': False,
 'antibiotics_Augmentin': True,
 'antibiotics_Clindamycin': False,
 'antibiotics_Invanz': False,
 'antibiotics_Kefsol': False,
 'antibiotics_Rocephin': False}